# ML5. Decision Trees

## 1. Подготовка данных

### 1.1 Импорты

In [1]:
!pip install catboost -q

In [2]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
import warnings
warnings.filterwarnings('ignore', message='Found unknown categories', category=UserWarning)
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from typing import List
import lightgbm as lgb
import catboost as cb
import xgboost as xgb

### 1.2. Загрузка данных

In [3]:
input_df = pd.read_csv('training.csv')
input_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

### 1.3. Очистка, сортировка и валидация данных по дате

In [4]:
input_df = input_df.drop(columns=['PRIMEUNIT', 'AUCGUART', 'RefId', 'Color', 'WheelTypeID', 'BYRNO', 'VNZIP1'])

input_df['PurchDate'] = pd.to_datetime(input_df['PurchDate'])
input_df = input_df.sort_values('PurchDate', ascending=True).reset_index(drop=True)

unique_dates = input_df['PurchDate'].dropna().unique()
n_dates = len(unique_dates)
third = n_dates // 3

train_dates = unique_dates[:third]
valid_dates = unique_dates[third:2*third]
test_dates = unique_dates[2*third:]

train_df = input_df[input_df['PurchDate'].isin(train_dates)]
y_train = train_df['IsBadBuy']

valid_df = input_df[input_df['PurchDate'].isin(valid_dates)]
y_valid = valid_df['IsBadBuy']

test_df = input_df[input_df['PurchDate'].isin(test_dates)]
y_test = test_df['IsBadBuy']

print(f"Train shape: {train_df.shape}")
print(f"Train dates: {train_dates.date.min()} - {train_dates.date.max()}\n")

print(f"Valid shape: {valid_df.shape}")
print(f"Valid dates: {valid_dates.date.min()} - {valid_dates.date.max()}\n")

print(f"Test shape: {test_df.shape}")
print(f"Test dates: {test_dates.date.min()} - {test_dates.date.max()}")

Train shape: (23059, 27)
Train dates: 2009-01-05 - 2009-09-01

Valid shape: (24104, 27)
Valid dates: 2009-09-02 - 2010-04-30

Test shape: (25820, 27)
Test dates: 2010-05-03 - 2010-12-30


### 1.4. Подготовка признаков

In [5]:
ohe_features = ['Auction', 'Transmission', 'WheelType', 'Nationality', 'Size', 'VNST', 'IsOnlineSale']
te_features = ['Make', 'Model', 'Trim', 'SubModel']
numeric_features = ['VehYear', 'VehicleAge', 'VehOdo', 'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'VehBCost', 'WarrantyCost']
cat_features = ohe_features + te_features

X_train = train_df[numeric_features + cat_features].copy()
X_val = valid_df[numeric_features + cat_features].copy()
X_test = test_df[numeric_features + cat_features].copy()


In [6]:
num = SimpleImputer(strategy='median')

ohe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))
    ])

te = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('te', TargetEncoder(
            target_type='binary',
            smooth=20,
            cv=5,
            shuffle=True,
            random_state=21
        ))
    ])

preprocessor = ColumnTransformer([
    ('num', num, numeric_features),
    ('ohe', ohe, ohe_features),
    ('te', te, te_features)
], remainder='drop')

## 2. Реализация классов

### 2.1 Node

In [7]:
class Node:
  def __init__(self, X=None, y=None, sample_indices=None, depth=0):
    self.X = X
    self.y = y
    self.sample_indices = sample_indices
    self.depth = depth
    self.left = None
    self.right = None
    self.feature_idx = None
    self.threshold = None
    self.prediction = None
    self.probabilities = None
    self.value = None

  def is_leaf(self):
    return self.left is None and self.right is None

  def gini_impurity(self):
    y_node = self.y[self.sample_indices]
    if len(y_node) == 0:
      return 0.0
    _, counts = np.unique(y_node, return_counts=True)
    probs = counts / len(y_node)
    return 1.0 - np.sum(probs ** 2)

  def std_dev(self):
    y_node = self.y[self.sample_indices]
    if len(y_node) == 0:
      return 0.0
    return np.std(y_node, ddof=0)

  @staticmethod
  def find_random_split(X, y, indices, criterion='gini', n_thresholds=5, feature_subset=None):
    n_samples = len(indices)
    n_features_total = X.shape[1]

    if feature_subset is None:
      feature_subset = np.arange(n_features_total)

    best_gain = -np.inf
    best_feature = None
    best_threshold = None

    parent = Node(X=X, y=y, sample_indices=indices)
    parent_metric = parent.gini_impurity() if criterion == 'gini' else parent.std_dev()

    for feat in feature_subset:
      feat_vals = X[indices, feat]
      feat_min, feat_max = feat_vals.min(), feat_vals.max()

      thresholds = np.random.uniform(feat_min, feat_max, n_thresholds)

      for thresh in thresholds:

        left_mask = feat_vals <= thresh
        n_left = np.sum(left_mask)
        n_right = n_samples - n_left
        if n_left == 0 or n_right == 0:
          continue

        left_indices = indices[left_mask]
        right_indices = indices[~left_mask]

        left_node = Node(X=X, y=y, sample_indices=left_indices)
        right_node = Node(X=X, y=y, sample_indices=right_indices)

        left_met = left_node.gini_impurity() if criterion == 'gini' else left_node.std_dev()
        right_met = right_node.gini_impurity() if criterion == 'gini' else right_node.std_dev()

        weighted_child_metric = (n_left / n_samples) * left_met + (n_right / n_samples) * right_met
        gain = parent_metric - weighted_child_metric

        if gain > best_gain:
          best_gain = gain
          best_feature = feat
          best_threshold = thresh

    return best_feature, best_threshold, best_gain

  @staticmethod
  def find_best_split(X, y, indices, criterion='gini', feature_subset=None):
    n_samples = len(indices)
    n_features_total = X.shape[1]

    if feature_subset is None:
      feature_subset = np.arange(n_features_total)

    best_gain = -np.inf
    best_feature = None
    best_threshold = None

    parent = Node(X=X, y=y, sample_indices=indices)
    parent_metric = parent.gini_impurity() if criterion == 'gini' else parent.std_dev()

    for feat in feature_subset:
      feat_vals = X[indices, feat]
      thresholds = np.unique(feat_vals)
      if len(thresholds) <= 1:
        continue

      mids = (thresholds[:-1] + thresholds[1:]) / 2.0

      for thresh in mids:
        left_mask = feat_vals <= thresh
        n_left = np.sum(left_mask)
        n_right = n_samples - n_left
        if n_left == 0 or n_right == 0:
          continue

        left_indices = indices[left_mask]
        right_indices = indices[~left_mask]

        left_node = Node(X=X, y=y, sample_indices=left_indices)
        right_node = Node(X=X, y=y, sample_indices=right_indices)

        left_met = left_node.gini_impurity() if criterion == 'gini' else left_node.std_dev()
        right_met = right_node.gini_impurity() if criterion == 'gini' else right_node.std_dev()

        weighted_child_metric = (n_left / n_samples) * left_met + (n_right / n_samples) * right_met
        gain = parent_metric - weighted_child_metric

        if gain > best_gain:
          best_gain = gain
          best_feature = feat
          best_threshold = thresh

    return best_feature, best_threshold, best_gain

### 2.2 DecisionTreeClassifier

In [8]:
class DTC:
  def __init__(self, max_depth=5, min_samples_split=2, splitter='best'):
    self.max_depth = max_depth
    self.min_samples_split = min_samples_split
    self.splitter = splitter
    self.root = None
    self.classes_ = None
    self.X_train = None
    self.y_train = None

  def _build_tree(self, indices, depth):
    node = Node(X=self.X_train, y=self.y_train, sample_indices=indices, depth=depth)
    y_node = self.y_train[indices]

    if (depth >= self.max_depth or
        len(indices) < self.min_samples_split or
        len(np.unique(y_node)) == 1):
      self._make_leaf(node)
      return node

    if hasattr(self, '_n_features_to_use') and self._n_features_to_use < self.X_train.shape[1]:
      n_total = self.X_train.shape[1]
      n_to_use = self._n_features_to_use
      feature_subset = np.random.choice(n_total, size=n_to_use, replace=False)
    else:
      feature_subset = None

    if self.splitter == 'best':
      feat, thresh, gain = Node.find_best_split(self.X_train, self.y_train, indices, criterion='gini', feature_subset=feature_subset)
    else:
      feat, thresh, gain = Node.find_random_split(self.X_train, self.y_train, indices, criterion='gini', feature_subset=feature_subset, n_thresholds=5)

    if feat is None or gain <= 1e-7:
      self._make_leaf(node)
      return node

    node.feature_idx = feat
    node.threshold = thresh

    feat_vals = self.X_train[indices, feat]
    left_mask = feat_vals <= thresh
    right_mask = ~left_mask

    node.left = self._build_tree(indices[left_mask], depth + 1)
    node.right = self._build_tree(indices[right_mask], depth + 1)

    return node

  def _make_leaf(self, node):
    y_node = self.y_train[node.sample_indices]
    classes, counts = np.unique(y_node, return_counts=True)
    node.prediction = classes[np.argmax(counts)]

    probs = np.zeros(self.n_classes)
    for i, c in enumerate(self.classes_):
      idx = np.where(classes == c)[0]
      if len(idx) > 0:
        probs[i] = counts[idx[0]]
    node.probabilities = probs / probs.sum() if probs.sum() > 0 else probs

  def _predict_one(self, x, node):
    if node.is_leaf():
      return node.prediction, node.probabilities
    if x[node.feature_idx] <= node.threshold:
      return self._predict_one(x, node.left)
    return self._predict_one(x, node.right)


  def fit(self, X, y):
    self.X_train = X.values if hasattr(X, 'values') else X
    self.y_train = np.asarray(y).ravel()
    self.classes_ = np.unique(self.y_train)
    self.n_classes = len(self.classes_)
    self.root = self._build_tree(indices=np.arange(len(self.y_train)), depth=0)
    return self

  def predict(self, X):
    return np.array([self._predict_one(x, self.root)[0] for x in X])

  def predict_proba(self, X):
    return np.array([self._predict_one(x, self.root)[1] for x in X])

### 2.2 DecisionTreeRegressor

In [9]:
class DTR:
  def __init__(self, max_depth=5, min_samples_split=2, splitter='best'):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.splitter = splitter
        self.root = None
        self.X_train = None
        self.y_train = None

  def fit(self, X, y):
      self.X_train = X
      self.y_train = y
      self.root = self._build_tree(indices=np.arange(len(y)), depth=0)
      return self

  def _make_leaf(self, node):
    y_node = self.y_train[node.sample_indices]
    node.prediction = np.mean(y_node)

  def _build_tree(self, indices, depth):
      node = Node(X=self.X_train, y=self.y_train, sample_indices=indices, depth=depth)
      y_node = self.y_train[indices]

      if (depth >= self.max_depth or
          len(indices) < self.min_samples_split or
          np.std(y_node) < 1e-7):
          self._make_leaf(node)
          return node

      if self.splitter == 'best':
          feat, thresh, gain = Node.find_best_split(self.X_train, self.y_train, indices, criterion='std')
      else:
          feat, thresh, gain = Node.find_random_split(self.X_train, self.y_train, indices, criterion='std')

      if feat is None or gain <= 1e-7:
          self._make_leaf(node)
          return node

      node.feature_idx = feat
      node.threshold = thresh

      feat_vals = self.X_train[indices, feat]
      left_mask = feat_vals <= thresh
      right_mask = ~left_mask

      node.left = self._build_tree(indices[left_mask], depth + 1)
      node.right = self._build_tree(indices[right_mask], depth + 1)
      return node

  def _make_leaf(self, node):
      y_node = self.y_train[node.sample_indices]
      node.prediction = np.mean(y_node)

  def _predict_one(self, x, node):
      if node.is_leaf():
          return node.prediction
      if x[node.feature_idx] <= node.threshold:
          return self._predict_one(x, node.left)
      return self._predict_one(x, node.right)

  def predict(self, X):
      return np.array([self._predict_one(x, self.root) for x in X])

## 3. Обучение собственной модели

In [10]:
my_dtc = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', DTC(max_depth=5, min_samples_split=10))
])
my_dtc.fit(X_train, y_train)
my_dtc_pred = my_dtc.predict_proba(X_val)[:, 1]
gini_my_dtc = 2 * roc_auc_score(y_valid, my_dtc_pred) - 1
print(f"Gini score for my DessisionTreeClassifier: {gini_my_dtc:.5f}")

Gini score for my DessisionTreeClassifier: 0.44772


## 4. Sklearn's DecisionTreeClassifier

In [11]:
sklearn_dtc = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', DecisionTreeClassifier(max_depth=5, min_samples_split=10))
])
sklearn_dtc.fit(X_train, y_train)
sklearn_dtc_pred = sklearn_dtc.predict_proba(X_val)[:, 1]
gini_sklearn_dtc = 2 * roc_auc_score(y_valid, sklearn_dtc_pred) - 1
print(f"Gini score for sklearn DessisionTreeClassifier: {gini_sklearn_dtc:.5f}")

Gini score for sklearn DessisionTreeClassifier: 0.44577


Результат получился практически идентичный

## 5. Реализация RandomForestClassifier

**Random Forest** (случайный лес) — ансамблевый алгоритм машинного обучения, который использует множество решающих деревьев для повышения точности предсказаний. Он применяется для задач классификации, регрессии, а также в некоторых случаях для кластеризации, отбора признаков и обнаружения аномалий.

In [12]:
class RFC:
  def __init__(
      self, n_estimators=100, max_depth=5, max_features='sqrt',
      min_samples_split=2, boostrap=True, random_state=None
  ):
    self.n_estimators = n_estimators
    self.max_depth = max_depth
    self.max_features = max_features
    self.min_samples_split = min_samples_split
    self.boostrap = boostrap
    self.random_state = random_state
    self.trees: List[DecisionTreeClassifier] = []
    self.oob_indices_ = []
    self.classes_ = None

  def _get_n_features_to_sample(self, n_total_features):
    if self.max_features == 'sqrt':
      return int(np.sqrt(n_total_features))
    elif self.max_features == 'log2':
      return int(np.log2(n_total_features))
    elif isinstance(self.max_features, int):
      return int(self.max_features * n_total_features)
    else:
      return n_total_features


  def fit(self, X, y):
    if self.random_state is not None:
      np.random.seed(self.random_state)

    X = np.asarray(X)
    y = np.asarray(y).ravel()
    self.classes_ = np.unique(y)
    n_samples, n_features = X.shape
    _n_features_to_use = self._get_n_features_to_sample(n_features)

    self.trees = []
    self.oob_indices_ = []

    for t in range(self.n_estimators):
      if self.boostrap:
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        oob_mask = np.ones(n_samples, dtype=bool)
        oob_mask[np.unique(indices)] = False
        self.oob_indices_.append(np.where(oob_mask)[0])
      else:
        indices = np.arange(n_samples)
        self.oob_indices_.append(np.array([]))

      X_boot = X[indices]
      y_boot = y[indices]

      tree = DecisionTreeClassifier(
          max_depth=self.max_depth,
          min_samples_split=self.min_samples_split,
          splitter='random'
      )

      tree._n_features_to_use = _n_features_to_use
      tree._original_n_features = n_features

      tree.fit(X_boot, y_boot)
      self.trees.append(tree)

    return self

  def predict(self, X):
    X = np.asarray(X)
    all_preds = np.array([tree.predict(X) for tree in self.trees])

    predictions = []
    for i in range(X.shape[0]):
      votes = all_preds[:, i]
      pred = self.classes_[np.argmax(np.bincount(votes, minlength=len(self.classes_)))]
      predictions.append(pred)
    return np.array(predictions)

  def predict_proba(self, X):
    X = np.asarray(X)
    all_proba = np.array([tree.predict_proba(X) for tree in self.trees])
    return np.mean(all_proba, axis=0)

In [13]:
rf = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RFC(
        n_estimators=100,
        max_depth=5,
        max_features='sqrt',
        min_samples_split=5,
        boostrap=True,
        random_state=21
    ))
])

rf.fit(X_train, y_train)
rf_pred = rf.predict_proba(X_val)[:, 1]
gini_rf = 2 * roc_auc_score(y_valid, rf_pred) - 1
print(f"Gini score for my RandomForestClassifier: {gini_rf:.5f}")

Gini score for my RandomForestClassifier: 0.48881


Как мы видим результат улучшился

## 6. Реализация GBDT

Деревья решений с градиентным бустингом (Gradient Boosted Decision Trees, GBDT) — это эффективный метод машинного обучения, используемый в основном для задач регрессии и классификации. Они объединяют прогнозы нескольких слабых учеников, обычно деревьев решений, для создания сильной прогностической модели.

Основная идея заключается в последовательном построении моделей, при этом каждая новая модель пытается исправить ошибки, допущенные предыдущими.

In [14]:
class GBDT:
  def __init__(self, learning_rate=0.1, max_depth=5, n_trees=100, max_features=None):
    self.max_depth= max_depth
    self.n_trees = n_trees
    self.max_features = max_features
    self.learning_rate = learning_rate
    self.trees = []
    self.base_score = None
    self.classes_ = np.array([0, 1])

  @staticmethod
  def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

  def fit(self, X, y):
    X = np.asarray(X)
    y = np.asarray(y).ravel()
    n_samples = len(y)

    pos_ratio = np.clip(y.mean(), 1e-15, 1 - 1e-15)
    self.base_score = np.log(pos_ratio / (1 - pos_ratio))
    F = np.full(n_samples, self.base_score)

    self.trees = []
    for t in range(self.n_trees):
      p = self.sigmoid(F)
      residuals = y - p # градиент бинарной кросс-энтропии
      tree = DecisionTreeRegressor(
          max_depth=self.max_depth,
          max_features=self.max_features
      )
      tree.fit(X, residuals)
      self.trees.append(tree)

      F += self.learning_rate * tree.predict(X)

    return self

  def predict_proba(self, X):
    X = np.asarray(X)
    F = np.full(X.shape[0], self.base_score)
    for tree in self.trees:
      F += self.learning_rate * tree.predict(X)
    prob_1 = self.sigmoid(F)
    return np.column_stack((1 - prob_1, prob_1))

  def predict(self, X):
    proba = self.predict_proba(X)
    return (proba[:, 1] >= 0.5).astype(int)


In [15]:
gbdt = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', GBDT(learning_rate=0.1, max_depth=5, n_trees=100))
])

In [16]:
gbdt.fit(X_train, y_train)
gbdt_pred = gbdt.predict_proba(X_val)[:, 1]
gini_gbdt = 2 * roc_auc_score(y_valid, gbdt_pred) - 1
print(f"Gini score for my GBDT: {gini_gbdt:.5f}")

Gini score for my GBDT: 0.48205


## 7. LightGBM, Catboost, XGBoost

### 7.1 LightGBM

Особенности модели:

* Leaf-wise стратегия построения деревьев. В отличие от традиционных алгоритмов бустинга, которые строят деревья по уровням (горизонтально), LightGBM выбирает для роста лист с максимальной дельта-потерей. Это позволяет снизить потери более значительно, чем при построении по уровням, что приводит к более высокой точности и более быстрой конвергенции.

* GOSS (Gradient-based One-Side Sampling). Метод выборки обучающих примеров на основе величины градиентов. Сохраняются все примеры с большими градиентами (которые хуже предсказываются моделью), а часть примеров с малыми градиентами выбирается случайно. Это уменьшает объём данных, необходимых для обучения, и ускоряет процесс.

* EFB (Exclusive Feature Bundling). Оптимизация, которая объединяет взаимоисключающие признаки (которые редко принимают ненулевые значения одновременно) в единые «пакеты». Особенно эффективна для разреженных данных, где множество признаков имеют нулевые значения. Позволяет сократить количество признаков без потери информации, ускоряет обучение и снижает потребление памяти.

* Обучение на основе гистограмм. Непрерывные значения признаков дискретизируются в гистограммы, что ускоряет обучение модели и позволяет быстро вычислять точки разделения для деревьев принятия решений. Это особенно эффективно для огромных наборов данных, так как устраняет необходимость в сортировке и сканировании всех точек данных.

* Обработка категориальных признаков. LightGBM имеет встроенную поддержку для работы с категориальными данными, что позволяет эффективно обрабатывать такие признаки без обширной предварительной обработки.

* Настраиваемые целевые и оценочные функции. Можно определять пользовательские функции потерь и метрики, что полезно для решения конкретных задач машинного обучения.

* Регуляризация. В LightGBM доступны параметры для регуляризации (например, lambda_l1, lambda_l2), которые помогают предотвратить переобучение.

* DART (Dropouts meet Multiple Additive Regression Trees). Метод регуляризации, который использует отсев (dropout) из нейронных сетей для деревьев решений. На каждом цикле обучения DART случайным образом исключает подмножество деревьев, что снижает переобучение и улучшает обобщающую способность модели.
geeksforgeeks.org

In [25]:
X_train_lgb = X_train.copy()
X_val_lgb = X_val.copy()

for col in numeric_features:
  median = X_train[col].median()
  X_train_lgb[col] = X_train_lgb[col].fillna(median)
  X_val_lgb[col] = X_val_lgb[col].fillna(median)

for col in cat_features:
  all_cats = pd.concat([X_train_lgb[col], X_val_lgb[col]]).astype(str).unique()
  cat_dtype = pd.CategoricalDtype(categories=all_cats, ordered=True)
  X_train_lgb[col] = X_train_lgb[col].fillna('Missing').astype(cat_dtype)
  X_val_lgb[col] = X_val_lgb[col].fillna('Missing').astype(cat_dtype)

In [26]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=5,
    min_data_in_leaf=25,
    feature_fraction=0.7,
    bagging_fraction=0.7,
    bagging_freq=5,
    min_split_gain=0.01,
    is_unbalance=True,
    random_state=21,
    verbose=-1,
    n_jobs=-1
)
lgb_model.fit(
    X_train_lgb, y_train,
    categorical_feature=cat_features,
    eval_set=[(X_val_lgb, y_valid)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
  )
lgb_pred = lgb_model.predict_proba(X_val_lgb)[:, 1]
gini_lgb = 2 * roc_auc_score(y_valid, lgb_pred) - 1
print(f"Gini score for my LGB: {gini_lgb:.5f}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[6]	valid_0's binary_logloss: 0.375078
Gini score for my LGB: 0.46443


### 7.2 Catboost

Особенности модели:

* Обработка категориальных признаков без предварительного кодирования. CatBoost автоматически определяет категориальные признаки в данных и использует метод упорядоченного кодирования (ordered boosting). При этом кодирование каждого объекта данных опирается на информацию из предыдущих объектов, что помогает избежать потери данных и переобучения. Также алгоритм использует счётчики для категориальных признаков — для каждой категории создаются числовые признаки, отражающие статистику по этим категориям (например, доля положительных и отрицательных классов внутри каждой категории).

* Автоматическая обработка пропущенных значений. Алгоритм присваивает пропущенным значениям преобладающий целочисленный признак или среднее значение вещественного признака.

* Борьба с переобучением. CatBoost использует несколько методов для предотвращения переобучения, включая мощную регуляризацию и усреднение. Среди специальных методов — градиентный шум и порядок бустинга.

* Скорость и производительность. В алгоритме реализованы оптимизации, которые делают обучение и предсказания более быстрыми по сравнению с аналогами. Среди них — поддержка многоядерной обработки и эффективное использование памяти. CatBoost оптимизирован для работы с большими наборами данных.

* Симметричные деревья решений. На каждом уровне глубины все узлы решений используют одно и то же условие разделения. Это ускоряет обучение, эффективно использует память и повышает устойчивость к небольшим изменениям в данных.



In [19]:
X_train_cb = X_train.copy()
X_val_cb = X_val.copy()
X_test_cb = X_test.copy()

for col in numeric_features:
    med = X_train_cb[col].median()
    X_train_cb[col] = X_train_cb[col].fillna(med)
    X_val_cb[col] = X_val_cb[col].fillna(med)
    X_test_cb[col] = X_test_cb[col].fillna(med)

for col in cat_features:
    X_train_cb[col] = X_train_cb[col].fillna('Missing').astype(str)
    X_val_cb[col] = X_val_cb[col].fillna('Missing').astype(str)
    X_test_cb[col] = X_test_cb[col].fillna('Missing').astype(str)

In [20]:
cb_model = cb.CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=5,
    l2_leaf_reg=3.0,
    bagging_temperature=0.5,
    random_strength=1.0,
    auto_class_weights='Balanced',
    eval_metric='AUC',
    loss_function='Logloss',
    verbose=False,
    random_state=21,
    thread_count=-1
)
cb_model.fit(
    X_train_cb, y_train,
    cat_features=cat_features,
    eval_set=(X_val_cb, y_valid),
    early_stopping_rounds=50,
    use_best_model=True
  )
cb_pred = cb_model.predict_proba(X_val_cb)[:, 1]
gini_cb = 2 * roc_auc_score(y_valid, cb_pred) - 1
print(f"Gini score for my Catboost: {gini_cb:.5f}")

Gini score for my Catboost: 0.49179


### 7.3 XGBoost

Особенности модели:

* Ансамблевое обучение. XGBoost строит модель путём последовательного добавления слабых моделей (обычно решающих деревьев). Каждое новое дерево фокусируется на исправлении ошибок, допущенных предыдущими деревьями.

* Регуляризация. В целевую функцию включены термины регуляризации L1 (Lasso) и L2 (Ridge), что помогает предотвратить переобучение.

* Гистограммный подход. Вместо сортировки всех значений признаков используются гистограммы, что ускоряет поиск оптимальных разбиений.

* Обработка разреженных данных. Алгоритм автоматически определяет оптимальное направление для пропущенных значений.

* Второй порядок оптимизации. XGBoost использует информацию о вторых производных (гессиан) функции потерь, что обеспечивает более точную аппроксимацию и быструю сходимость.

* Гибкая настройка параметров. Параметры XGBoost делятся на три категории: общие (определяют общую функциональность модели), параметры бустинга (настраивают процесс каждого раунда бустинга) и параметры задачи обучения (определяют функцию потерь и цель модели).

In [21]:
xgb_model = Pipeline([
    ('prep', preprocessor),
    ('clf', xgb.XGBClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=4,
        gamma=0.5,
        reg_lambda=2.0,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        eval_metric='auc',
        random_state=21,
        n_jobs=-1,
        verbosity=0,
        enable_categorial=True
    ))
])

In [22]:
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict_proba(X_val)[:, 1]
gini_xgb = 2 * roc_auc_score(y_valid, xgb_pred) - 1
print(f"Gini score for XBGoost: {gini_xgb:.5f}")

Gini score for XBGoost: 0.45593


| Ситуация | Модель |
|---------------|--------------|
| Много категорий, временной сплит, мало времени на тюнинг | **CatBoost** |
| >500k строк, ограничен RAM/время, нужна скорость | **LightGBM** |
| Нужен кастомный лосс, полный контроль регуляризации | **XGBoost** |
| Обучение, демонстрация, малые данные | **Classic GBDT / RF** |

## 8. Лучшая модель

Лучший результат показала модель CatBoost т.к. она хорошо подходит под наши данные(категориальные признаки с высокой кардинальностью, коррелирующие числовые признаки, дисбаланс меток и т.д.)

In [23]:
score_df = pd.DataFrame({
    'model': ['DecisionTreeClassifier', 'RandomForestClassifier', 'XGBoost', 'CatBoost', 'LightGBM'],
    'Gini score': [gini_sklearn_dtc, gini_rf, gini_xgb, gini_cb, gini_lgb]
})
score_df.sort_values('Gini score', ascending=False).reset_index(drop=True)

,model,Gini score
0,CatBoost,0.491790
1,RandomForestClassifier,0.488807
2,LightGBM,0.464432
3,XGBoost,0.455931
4,DecisionTreeClassifier,0.445765


Оценим нашу лучшую модель на тестовых данных:

In [24]:
cb_pred_train = cb_model.predict_proba(X_train_cb)[:, 1]
gini_cb_train = 2 * roc_auc_score(y_train, cb_pred_train) - 1
cb_pred_test = cb_model.predict_proba(X_test_cb)[:, 1]
gini_cb_test = 2 * roc_auc_score(y_test, cb_pred_test) - 1
print(f"CatBoost results:\n")

best_model_score_df = pd.DataFrame({
    'set': ['train', 'valid', 'test'],
    'gini score': [gini_cb_train, gini_cb, gini_cb_test]
})
best_model_score_df

CatBoost results:



,set,gini score
0,train,0.555528
1,valid,0.491790
2,test,0.497723


Как мы видим модель стабильно работает на всех данных, что означает, что она не переобучена и хорошо предсказывает целевое значение.